# Bronze Ingestion: ENTSO-E Day-Ahead Prices

Fetches Finnish day-ahead electricity prices from the ENTSO-E Transparency 
Platform API and writes them to a Bronze Delta table without transformation.

**Domain:** Finland (10YFI-1--------U)<br>
**Source:** https://web-api.tp.entsoe.eu/api<br>
**Output:** electricity_project.bronze.entsoe_dayahead_prices<br>

In [0]:
import requests
import time
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

Retrieves the ENTSO-E API key from Databricks Secrets.

In [0]:
api_key = dbutils.secrets.get(scope="electricity-project", key="entsoe-api-key")

Defines the ENTSO-E API base URL and Finland's domain code (EIC), used in 
all requests to this API.

In [0]:
ENTSOE_BASE_URL = "https://web-api.tp.entsoe.eu/api"
FINLAND_DOMAIN = "10YFI-1--------U"

Defines the XML namespace used in ENTSO-E's response, required by 
ElementTree to correctly locate elements within the document.

In [0]:
ENTSOE_NS = {"ns": "urn:iec62325.351:tc57wg16:451-3:publicationdocument:7:3"}

Sends a single request to the ENTSO-E API for a given time period and 
returns the raw XML response as text, without parsing or pagination.

In [0]:
def fetch_entsoe_raw(period_start, period_end, api_key):
    params = {
        "securityToken": api_key,
        "documentType": "A44",
        "in_Domain": FINLAND_DOMAIN,
        "out_Domain": FINLAND_DOMAIN,
        "periodStart": period_start,
        "periodEnd": period_end
    }
    response = requests.get(ENTSOE_BASE_URL, params=params)
    response.raise_for_status()
    return response.text

Converts ENTSO-E's ISO 8601 duration format (e.g. "PT15M") into an integer 
number of minutes, used to calculate each point's actual timestamp.

In [0]:
def parse_resolution_minutes(resolution_str):
    return int(resolution_str.replace("PT", "").replace("M", ""))

Parses the raw ENTSO-E XML into a list of rows, one per 15-minute interval, 
calculating each timestamp from the period start and resolution, and 
forward-filling prices for positions missing from the XML.

In [0]:
def parse_entsoe_prices(xml_text, ns):
    root = ET.fromstring(xml_text)
    rows = []

    for timeseries in root.findall("ns:TimeSeries", ns):
        period = timeseries.find("ns:Period", ns)
        start_str = period.find("ns:timeInterval/ns:start", ns).text
        end_str = period.find("ns:timeInterval/ns:end", ns).text
        resolution_minutes = parse_resolution_minutes(period.find("ns:resolution", ns).text)
        period_start = datetime.strptime(start_str, "%Y-%m-%dT%H:%MZ")
        period_end = datetime.strptime(end_str, "%Y-%m-%dT%H:%MZ")
        total_positions = int((period_end - period_start).total_seconds() / 60 / resolution_minutes)

        points = period.findall("ns:Point", ns)
        price_by_position = {
            int(p.find("ns:position", ns).text): float(p.find("ns:price.amount", ns).text)
            for p in points
        }

        last_price = None
        for position in range(1, total_positions + 1):
            if position in price_by_position:
                last_price = price_by_position[position]
            timestamp = period_start + timedelta(minutes=(position - 1) * resolution_minutes)
            rows.append({
                "timestamp": timestamp.strftime("%Y-%m-%dT%H:%M:%SZ"),
                "price_eur_mwh": last_price
            })

    return rows

Fetches prices for a given date range by looping through it in ~31-day 
chunks, since ENTSO-E limits a single response to 100 TimeSeries blocks. 
Deduplicates rows by timestamp, since chunk boundaries can cause the same 
day to be returned twice.

In [0]:
def fetch_entsoe_prices(start_date, end_date, api_key):
    all_rows = []
    current = start_date
    while current < end_date:
        next_month = min(current + timedelta(days=31), end_date)
        period_start = current.strftime("%Y%m%d%H%M")
        period_end = next_month.strftime("%Y%m%d%H%M")
        xml_text = fetch_entsoe_raw(period_start, period_end, api_key)
        rows = parse_entsoe_prices(xml_text, ENTSOE_NS)
        all_rows.extend(rows)
        current = next_month
        time.sleep(1)

    deduplicated = {row["timestamp"]: row for row in all_rows}
    return list(deduplicated.values())

Fetches (2025-10-01 to 2026-09-01) of Finnish day-ahead prices, 
matching the same period used for the Fingrid consumption and production data.

In [0]:
start_date = datetime(2025, 10, 1)
end_date = datetime(2026, 9, 1)

entsoe_data = fetch_entsoe_prices(start_date, end_date, api_key)
print(len(entsoe_data))

32256


Writes the deduplicated ENTSO-E price data as a Bronze Delta table, using 
an explicit schema to avoid type inference conflicts.

In [0]:
entsoe_schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("price_eur_mwh", DoubleType(), True)
])

entsoe_df = spark.createDataFrame(entsoe_data, schema=entsoe_schema)
entsoe_df.write.format("delta").mode("overwrite").saveAsTable("electricity_project.bronze.entsoe_dayahead_prices")

In [0]:
display(spark.table("electricity_project.bronze.entsoe_dayahead_prices").limit(5))

timestamp,price_eur_mwh
2026-07-21T22:00:00Z,21.55
2026-07-21T22:15:00Z,20.45
2026-07-21T22:30:00Z,19.11
2026-07-21T22:45:00Z,16.58
2026-07-21T23:00:00Z,20.18
